In [ ]:
%matplotlib inline


import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
from PIL import Image
from tempfile import TemporaryDirectory
import mlflow
from mlflow.models import infer_signature
import GPUtil
from sklearn.metrics import f1_score,precision_score,recall_score
import torchvision.transforms.functional as F

# plt.ion()   # interactive mode

TODO edit this function to have a bigger batch size
image_datasets['train'].__getitem__

Try with uncropped images

Try seperate models for each part of the prediction that feed into each other

In [ ]:
host = "127.0.0.1"
port = "8080"
mlflow.set_tracking_uri(uri=F"http://{host}:{port}")

In [ ]:
data_dir = '../../../data/Car Labelling/Standford Dataset/'
os.path.exists(data_dir)

In [ ]:
image_size = 384
normalized_mean = [0.485, 0.456, 0.406]
normalized_std = [0.229, 0.224, 0.225]


params = {
    'batch_size': 128,
    'num_workers': 0,
    'epochs': 50,
    'frozen_layers': True,
    'pre_cropped': True,
    'resized': True,
    'padding': True,
    # 'interpolation': str(Image.BILINEAR),
    'interpolation': 'NA',
    'new_image_size': str(image_size),
    'normalized': True,
    'normalized_mean': '[0.485, 0.456, 0.406]',
    'normalized_std': '[0.229, 0.224, 0.225]',
    'optim': 'SGD',
    'lr': 0.5,
    'momentum': 0.9,
    'lr_scheduled_used': True,
    'lr_sched_step_size': 10,
    'lr_sched_gamma': 0.8
}

model_type = "EfficientNet"
model_name = "efficientnet_v2_s"

model_use = "Car Labelling"

experiment_tags = {
    "dataset": F"Standford",
    "Train": "0.75",
    "model_use": model_use,
    "model_name": model_name
}

exp_name = F"{model_use}. Standford Dataset. {model_type} Only."

In [ ]:
experiment_tags

In [ ]:
class ResizeWithPad:
    def __init__(self, target_size, fill_value=0, padding_mode='constant'):
        self.target_size = target_size  # (height, width)
        self.fill_value = fill_value
        self.padding_mode = padding_mode

    def __call__(self, img):
        # Get original image dimensions
        img_w, img_h = img.size
        target_h, target_w = self.target_size

        # Calculate scaling factor to fit within target_size while maintaining aspect ratio
        scale = min(target_w / img_w, target_h / img_h)

        # Calculate new dimensions after scaling
        new_w, new_h = int(img_w * scale), int(img_h * scale)

        # Resize the image
        img = F.resize(img, (new_h, new_w))

        # Calculate padding
        pad_left = (target_w - new_w) // 2
        pad_right = target_w - new_w - pad_left
        pad_top = (target_h - new_h) // 2
        pad_bottom = target_h - new_h - pad_top

        padding = (pad_left, pad_top, pad_right, pad_bottom)

        # Apply padding
        img = F.pad(img, padding, fill=self.fill_value, padding_mode=self.padding_mode)

        return img

In [ ]:
transformations = []

if params['resized']:
    if params['padding']:
        transformations.append(ResizeWithPad(target_size=(image_size, image_size)))
    else:
        transformations.append(transforms.Resize((image_size,image_size), interpolation=int(params['interpolation'])))

transformations.append(transforms.ToTensor())

if params['normalized']:
    transformations.append(transforms.Normalize(normalized_mean, normalized_std))

data_transforms = {
    'train': transforms.Compose(transformations),
    'val': transforms.Compose(transformations),
}

image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x),
                                          data_transforms[x])
                  for x in ['train', 'val']}


In [ ]:
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=params['batch_size'],
                                             shuffle=True, num_workers=params['num_workers'])
              for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")


In [ ]:
print(dataset_sizes)

In [ ]:
# inputs, classes = next(iter(dataloaders['train']))

In [ ]:
def imshow(inp, title=None):
    """Display image for Tensor."""
    # inp = inp.numpy().transpose((1, 2, 0))
    # mean = np.array([0.485, 0.456, 0.406])
    # std = np.array([0.229, 0.224, 0.225])
    # inp = std * inp + mean
    # inp = np.clip(inp, 0, 1)
    # inp = data_transforms['train'](inp.numpy().transpose((1, 2, 0))).T
    plt.imshow(inp.permute(1, 2, 0))
    if title is not None:
        plt.title(title)
    plt.pause(0.001)  # pause a bit so that plots are updated


# Get a batch of training data
# inputs, classes = next(iter(dataloaders['train']))

# # Make a grid from batch
# out = torchvision.utils.make_grid(inputs)

# imshow(out, title=[class_names[x] for x in classes])

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs):
    since = time.time()

    # Create a temporary directory to save training checkpoints
    with TemporaryDirectory() as tempdir:
        best_model_params_path = os.path.join(tempdir, 'best_model_params.pt')

        torch.save(model.state_dict(), best_model_params_path)
        metrics = {}
        metrics['best_acc'] = 0
        epoch_losses = []
        epoch_accs = []


        for epoch in range(num_epochs):
            y_preds = []
            y_labels = []
            print(f'Epoch {epoch}/{num_epochs - 1}')
            print('-' * 10)

            # Each epoch has a training and validation phase
            for phase in ['train', 'val']:
                if phase == 'train':
                    model.train()  # Set model to training mode
                else:
                    model.eval()   # Set model to evaluate mode

                if params['batch_size'] == 1:
                    for module in model.modules():
                        if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                            module.eval()

                running_loss = 0.0
                running_corrects = 0

                # Iterate over data.
                for inputs, labels in dataloaders[phase]:
                    inputs = inputs.to(device)
                    labels = labels.to(device)

                    # zero the parameter gradients
                    optimizer.zero_grad()

                    # forward
                    # track history if only in train
                    with torch.set_grad_enabled(phase == 'train'):
                        outputs = model(inputs)
                        _, preds = torch.max(outputs, 1)
                        loss = criterion(outputs, labels)

                        # backward + optimize only if in training phase
                        if phase == 'train':
                            loss.backward()
                            optimizer.step()

                    # statistics
                    running_loss += loss.item() * inputs.size(0)
                    running_corrects += torch.sum(preds == labels.data)
                    y_preds.extend(preds.cpu())
                    y_labels.extend(labels.data.cpu())

                if phase == 'train':
                    scheduler.step()

                epoch_loss = running_loss / dataset_sizes[phase]
                epoch_acc = running_corrects.double() / dataset_sizes[phase]
                epoch_losses.append(epoch_loss)
                epoch_accs.append(epoch_acc.item())


                print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

                # deep copy the model
                if phase == 'val' and epoch_acc > metrics['best_acc']:
                    metrics['best_acc'] = epoch_acc
                    torch.save(model.state_dict(), best_model_params_path)
                    metrics['best_f1'] = f1_score(y_labels, y_preds, average='weighted')
                    metrics['best_precision'] = precision_score(y_labels, y_preds, average='weighted')
                    metrics['best_recall'] = recall_score(y_labels, y_preds, average='weighted')

            print()

        metrics['time_elapsed'] = time.time() - since
        time_elapsed = time.time() - since
        print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
        best_acc = metrics['best_acc']
        print(f'Best val Acc: {best_acc}')

        # load best model weights
        model.load_state_dict(torch.load(best_model_params_path, weights_only=True))
    return model, metrics, epoch_losses, epoch_accs

In [ ]:
def __get_metrics( metrics):
    gpu = GPUtil.getGPUs()[0]

    metrics['GPU_memory'] = gpu.memoryUsed
    return metrics

In [ ]:
model = models.efficientnet_v2_s(weights='IMAGENET1K_V1')

In [ ]:

model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(class_names))

model = model.to(device)

if params['frozen_layers']:
    for param in model.parameters():
        param.requires_grad = False
        
    for param in model.classifier.parameters():
        param.requires_grad = True

    optimizer_ft = optim.SGD(filter(lambda p: p.requires_grad, model.parameters()), lr=params['lr'], momentum=params['momentum'])
else:
    optimizer_ft = optim.SGD(model.parameters(), lr=params['lr'], momentum=params['momentum'])

criterion = nn.CrossEntropyLoss()

# Decay LR by a factor of 0.1 every 7 epochs
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=params['lr_sched_step_size'], gamma=params['lr_sched_gamma'])
    

In [ ]:
mlflow.set_experiment(exp_name)

trained_model, metrics, epoch_losses, epoch_accs = train_model(model,
                                                    criterion,
                                                    optimizer_ft, 
                                                    exp_lr_scheduler,
                                                    params['epochs'])
# experiment_desc = F"Dataset: Dataset_2/{params['dataset']}.\nModel Use: {model_use} \nNumber of Samples: {num_samples} "

# run_name = F"{params['model_name']}-{params['dataset']}-ccm:{params['car_choice_metric']}"

with mlflow.start_run(tags=experiment_tags):
    # Log the hyperparameters
    mlflow.log_params(params)

    metrics = __get_metrics(metrics)

    # Log the metrics
    for key,val in metrics.items():
        mlflow.log_metric(key, val)

    # signature = infer_signature(datasets['train'], model.forward(datasets['train']))
    model_info = mlflow.pytorch.log_model(
        pytorch_model=model,
        name=model_name,
        # signature=signature,
        # input_example=datasets['train']
    )

    mlflow.set_logged_model_tags(model_info.model_id,params)


In [ ]:
epoch_loss_train = []
epoch_loss_val = []

for i,val in enumerate(epoch_losses):
    if i%2==0:
        epoch_loss_train.append(val)
    else:
        epoch_loss_val.append(val)


In [ ]:
plt.plot(epoch_loss_train,label= 'Train Loss')
plt.plot(epoch_loss_val,label= 'Val Loss')
plt.legend()
plt.title('Loss Graph')

plt.show()

In [ ]:
epoch_accs_train = []
epoch_accs_val = []

for i,val in enumerate(epoch_accs):
    if i%2==0:
        epoch_accs_train.append(val.item())
    else:
        epoch_accs_val.append(val.item())


In [ ]:
plt.plot(epoch_accs_train,label= 'Train Acc')
plt.plot(epoch_accs_val,label= 'Val Acc')
plt.legend()
plt.title('Accuracy Graph')

plt.show()